In [5]:
# ======================================================================
# REVIEWER 1 / PROBLEM 5
# FINAL ONE-CELL GOOGLE COLAB CODE
#
# EXTERNAL HUMAN-EVALUATION MORPHOLOGY AUDIT
#
# PURPOSE
# ----------------------------------------------------------------------
# 1. Verify that the 200 independently prepared human-evaluation
#    questions do not occur in the main ~15,000-question corpus.
#
# 2. Reproduce the main Clean corpus:
#       raw records      = 14,991
#       unique Clean Qs  = 14,689
#
# 3. Apply the same deterministic Relational CSE resources/rules to:
#       a) Human200
#       b) MainCorpus14689
#
# 4. Preserve punctuation/operators/code notation exactly while
#    inserting CSE markers only into alphabetic word spans.
#
# 5. Report reconstruction as a DIAGNOSTIC only.
#    Exact surface reconstruction is NOT required for the morphology
#    complexity analysis because the original clean questions remain
#    unchanged and CSE is only an internal preprocessing representation.
#
# 6. Compare SIX morphology-complexity metrics:
#       - CSE boundaries per question
#       - CSE boundaries per alphabetic word
#       - morphologically complex-word proportion
#       - mean alphabetic word length
#       - maximum CSE boundaries per word
#       - alphabetic words per question
#
# 7. Statistics:
#       - descriptive statistics
#       - delta = Human200 - MainCorpus14689
#       - independent bootstrap 95% CI, 10,000 replicates
#       - two-sided independent permutation test, 10,000 permutations
#       - Hedges' g
#       - Holm correction across six primary morphology tests
#
# 8. Save all outputs and print a final block for ChatGPT.
#
# REQUIRED FILES
# ----------------------------------------------------------------------
# 1. 200 сұрақ.json
# 2. baseline_15000.json
# 3. qaz_stems_unik_edit.xlsx
# 4. qaz_jurnaks.xls
# 5. qaz_endings_seg.xls
# 6. stop_words.txt
#
# ======================================================================


# ======================================================================
# 0. INSTALL / IMPORT
# ======================================================================

import os
import sys
import re
import json
import math
import random
import hashlib
import zipfile
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")


def ensure_package(package, import_name=None):
    import_name = (
        import_name
        or package.split("==")[0].replace("-", "_")
    )

    try:
        __import__(import_name)

    except Exception:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package,
            ]
        )


print("=" * 100)
print("INSTALLING / CHECKING PACKAGES")
print("=" * 100)

ensure_package("numpy")
ensure_package("pandas")
ensure_package("scipy")
ensure_package("openpyxl")
ensure_package("xlrd==2.0.1", "xlrd")


import numpy as np
import pandas as pd
import openpyxl
import xlrd

from scipy import stats


try:
    from google.colab import files
    IN_COLAB = True

except Exception:
    files = None
    IN_COLAB = False


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 360)
pd.set_option("display.max_colwidth", 180)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.9f}"
)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

SEED = 42
STAT_SEED = 20260901

N_BOOT = 10_000
N_PERM = 10_000
ALPHA = 0.05


EXPECTED = {
    "human_questions": 200,

    "baseline_records": 14991,
    "baseline_unique_clean": 14689,

    "stems_entries": 103624,
    "stems_unique": 103623,

    "jurnaq_entries": 146,
    "jurnaq_unique": 101,

    "ending_mappings": 3316,
    "ending_surfaces_unique": 3030,

    "stopword_entries": 189,
    "stopword_unique": 188,
}


OUT_DIR = Path(
    "/content/reviewer1_problem5_final"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ZIP_PATH = Path(
    "/content/reviewer1_problem5_final_outputs.zip"
)


random.seed(SEED)
np.random.seed(SEED)


# ======================================================================
# 2. FILE DISCOVERY
# ======================================================================

FILE_SPECS = {
    "human": {
        "exact": [
            "200 сұрақ.json",
        ],
        "patterns": [
            "*200*сұрақ*.json",
            "*200*.json",
        ],
    },

    "baseline": {
        "exact": [
            "baseline_15000.json",
            "baseline_15000(3).json",
        ],
        "patterns": [
            "baseline_15000*.json",
        ],
    },

    "stems": {
        "exact": [
            "qaz_stems_unik_edit.xlsx",
            "qaz_stems_unik_edit(2).xlsx",
        ],
        "patterns": [
            "qaz_stems_unik_edit*.xlsx",
        ],
    },

    "jurnaqs": {
        "exact": [
            "qaz_jurnaks.xls",
            "qaz_jurnaks(2).xls",
        ],
        "patterns": [
            "qaz_jurnaks*.xls",
        ],
    },

    "endings": {
        "exact": [
            "qaz_endings_seg.xls",
            "qaz_endings_seg(2).xls",
        ],
        "patterns": [
            "qaz_endings_seg*.xls",
        ],
    },

    "stopwords": {
        "exact": [
            "stop_words.txt",
            "stop_words (1)(2).txt",
        ],
        "patterns": [
            "*stop_words*.txt",
            "*stop*words*.txt",
        ],
    },
}


def find_one_file(spec):
    roots = [
        Path("/content"),
        Path("."),
    ]

    # Exact names first
    for root in roots:
        for name in spec["exact"]:
            candidate = root / name

            if candidate.is_file():
                return str(
                    candidate.resolve()
                )

    # Then wildcard patterns
    found = []

    for root in roots:
        for pattern in spec["patterns"]:
            found.extend(
                [
                    p
                    for p in root.glob(pattern)
                    if p.is_file()
                ]
            )

    unique = []
    seen = set()

    for p in found:
        rp = str(
            p.resolve()
        )

        if rp not in seen:
            unique.append(rp)
            seen.add(rp)

    if unique:
        return unique[0]

    return ""


def discover_files():
    return {
        key: find_one_file(spec)
        for key, spec
        in FILE_SPECS.items()
    }


paths = discover_files()


missing = [
    key
    for key, value
    in paths.items()
    if not value
]


if missing and IN_COLAB:

    print("\n" + "=" * 100)
    print("UPLOAD REQUIRED FILES")
    print("=" * 100)

    print(
        "1. 200 сұрақ.json\n"
        "2. baseline_15000.json\n"
        "3. qaz_stems_unik_edit.xlsx\n"
        "4. qaz_jurnaks.xls\n"
        "5. qaz_endings_seg.xls\n"
        "6. stop_words.txt\n"
    )

    files.upload()

    paths = discover_files()


missing = [
    key
    for key, value
    in paths.items()
    if not value
]


if missing:
    raise FileNotFoundError(
        "Missing required files: "
        +
        ", ".join(missing)
    )


print("\n" + "=" * 100)
print("SELECTED INPUT FILES")
print("=" * 100)

for key, path in paths.items():
    print(
        f"{key:<12} -> {path}"
    )


# ======================================================================
# 3. INPUT FILE HASHES
# ======================================================================

def sha256_file(
    path,
    block_size=1024 * 1024
):
    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                block_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


file_hashes = {
    key: sha256_file(path)
    for key, path in paths.items()
}


# ======================================================================
# 4. TEXT NORMALIZATION
#
# Conservative normalization for corpus matching only.
# It does NOT use fuzzy matching.
# ======================================================================

_punct_left = re.compile(
    r"\s+([.,!?;:%)\]\}])"
)

_punct_right = re.compile(
    r"([(\[\{])\s+"
)

_multi_space = re.compile(
    r"\s+"
)


def norm_space_punct(text):

    text = str(text)

    text = text.replace(
        " - ",
        "-"
    )

    text = _punct_left.sub(
        r"\1",
        text
    )

    text = _punct_right.sub(
        r"\1",
        text
    )

    text = _multi_space.sub(
        " ",
        text
    ).strip()

    return text


def clean_view(text):

    text = str(text)

    text = text.replace(
        "@@ ",
        ""
    )

    text = text.replace(
        "@@",
        ""
    )

    return norm_space_punct(
        text
    )


# ======================================================================
# 5. ROBUST BASELINE JSON LOADER
# ======================================================================

def normalize_qa_records(data):

    if not isinstance(
        data,
        list
    ):
        raise ValueError(
            "QA data must resolve to a list."
        )

    output = []

    for item in data:

        if not isinstance(
            item,
            dict
        ):
            continue

        question = str(
            item.get("question")
            or item.get("instruction")
            or ""
        ).strip()

        answer = str(
            item.get("answer")
            or item.get("response")
            or ""
        ).strip()

        if question:

            output.append(
                {
                    "question":
                        question,

                    "answer":
                        answer,
                }
            )

    if not output:
        raise ValueError(
            "No QA records found."
        )

    return output


def load_baseline_qa(path):

    text = Path(path).read_text(
        encoding="utf-8",
        errors="ignore"
    ).strip()

    if not text:
        raise ValueError(
            "Baseline JSON is empty."
        )

    # --------------------------------------------------
    # Standard JSON
    # --------------------------------------------------

    try:

        parsed = json.loads(
            text
        )

        if isinstance(
            parsed,
            list
        ):
            return normalize_qa_records(
                parsed
            )

        if isinstance(
            parsed,
            dict
        ):

            for key in [
                "data",
                "records",
                "items",
                "qa",
                "questions",
            ]:

                if isinstance(
                    parsed.get(key),
                    list
                ):

                    return normalize_qa_records(
                        parsed[key]
                    )

    except Exception:
        pass


    # --------------------------------------------------
    # JSONL attempt
    # --------------------------------------------------

    try:

        objects = []

        for line in text.splitlines():

            line = (
                line.strip()
                .rstrip(",")
            )

            if not line:
                continue

            objects.append(
                json.loads(
                    line
                )
            )

        if objects:

            return normalize_qa_records(
                objects
            )

    except Exception:
        pass


    # --------------------------------------------------
    # Robust object scanner
    # --------------------------------------------------

    objects = []

    buffer = []
    depth = 0
    in_string = False
    escaped = False
    started = False


    for ch in text:

        if not started:

            if ch == "{":

                started = True
                depth = 1
                buffer = ["{"]

            continue


        buffer.append(
            ch
        )


        if in_string:

            if escaped:
                escaped = False

            elif ch == "\\":
                escaped = True

            elif ch == '"':
                in_string = False


        else:

            if ch == '"':
                in_string = True

            elif ch == "{":
                depth += 1

            elif ch == "}":

                depth -= 1

                if depth == 0:

                    raw = "".join(
                        buffer
                    )

                    try:

                        obj = json.loads(
                            raw
                        )

                        if isinstance(
                            obj,
                            dict
                        ):
                            objects.append(
                                obj
                            )

                    except Exception:
                        pass

                    buffer = []
                    started = False


    return normalize_qa_records(
        objects
    )


# ======================================================================
# 6. HUMAN 200 LOADER
# ======================================================================

def load_human_questions(path):

    text = Path(path).read_text(
        encoding="utf-8-sig",
        errors="ignore"
    ).strip()

    if not text:
        raise ValueError(
            "Human file is empty."
        )

    try:

        parsed = json.loads(
            text
        )

        if isinstance(
            parsed,
            list
        ):

            questions = []

            for item in parsed:

                if isinstance(
                    item,
                    str
                ):

                    q = item.strip()

                elif isinstance(
                    item,
                    dict
                ):

                    q = str(
                        item.get("question")
                        or item.get("text")
                        or item.get("instruction")
                        or ""
                    ).strip()

                else:
                    q = ""

                if q:
                    questions.append(
                        q
                    )

            if questions:
                return questions

        elif isinstance(
            parsed,
            dict
        ):

            for key in [
                "questions",
                "items",
                "data",
                "records",
            ]:

                value = parsed.get(
                    key
                )

                if isinstance(
                    value,
                    list
                ):

                    temp_path = None

                    questions = []

                    for item in value:

                        if isinstance(
                            item,
                            str
                        ):

                            q = item.strip()

                        elif isinstance(
                            item,
                            dict
                        ):

                            q = str(
                                item.get("question")
                                or item.get("text")
                                or item.get("instruction")
                                or ""
                            ).strip()

                        else:
                            q = ""

                        if q:
                            questions.append(
                                q
                            )

                    if questions:
                        return questions

    except Exception:
        pass


    # Plain one-question-per-line fallback

    return [
        line.strip()
        for line
        in text.splitlines()
        if line.strip()
    ]


baseline_rows = load_baseline_qa(
    paths["baseline"]
)

human_questions = load_human_questions(
    paths["human"]
)


# ======================================================================
# 7. QUESTION DATA AUDIT
# ======================================================================

print("\n" + "=" * 100)
print("QUESTION DATA AUDIT")
print("=" * 100)

print(
    "Baseline raw records       =",
    f"{len(baseline_rows):,}"
)

print(
    "Human-evaluation questions =",
    f"{len(human_questions):,}"
)


if (
    len(
        baseline_rows
    )
    !=
    EXPECTED[
        "baseline_records"
    ]
):

    raise RuntimeError(
        "STOP: baseline record count mismatch. "
        f"Actual={len(baseline_rows)}, "
        f"expected={EXPECTED['baseline_records']}."
    )


if (
    len(
        human_questions
    )
    !=
    EXPECTED[
        "human_questions"
    ]
):

    raise RuntimeError(
        "STOP: human question count mismatch. "
        f"Actual={len(human_questions)}, "
        f"expected={EXPECTED['human_questions']}."
    )


# ----------------------------------------------------------------------
# Reproduce unique Clean corpus
# ----------------------------------------------------------------------

baseline_unique_map = {}


for row in baseline_rows:

    question = clean_view(
        row["question"]
    )

    if (
        question
        and
        question
        not in
        baseline_unique_map
    ):

        baseline_unique_map[
            question
        ] = row


baseline_unique_questions = list(
    baseline_unique_map.keys()
)


human_norm_questions = [
    clean_view(
        q
    )
    for q
    in human_questions
]


human_unique_exact = len(
    set(
        human_norm_questions
    )
)


human_unique_casefold = len(
    set(
        q.casefold()
        for q
        in human_norm_questions
    )
)


print(
    "Unique Clean corpus Qs     =",
    f"{len(baseline_unique_questions):,}"
)

print(
    "Human unique, exact        =",
    human_unique_exact
)

print(
    "Human unique, casefold     =",
    human_unique_casefold
)


if (
    len(
        baseline_unique_questions
    )
    !=
    EXPECTED[
        "baseline_unique_clean"
    ]
):

    raise RuntimeError(
        "STOP: unique Clean corpus mismatch. "
        f"Actual={len(baseline_unique_questions)}, "
        f"expected={EXPECTED['baseline_unique_clean']}."
    )


if (
    human_unique_exact
    !=
    EXPECTED[
        "human_questions"
    ]
):

    raise RuntimeError(
        "STOP: human question set is not "
        "200 unique normalized questions."
    )


# ======================================================================
# 8. OVERLAP / LEAKAGE AUDIT
#
# No fuzzy matching.
# ======================================================================

corpus_exact_set = set(
    baseline_unique_questions
)


corpus_casefold_set = set(
    q.casefold()
    for q
    in baseline_unique_questions
)


overlap_rows = []


for idx, question in enumerate(
    human_norm_questions,
    start=1
):

    exact_match = (
        question
        in
        corpus_exact_set
    )

    casefold_match = (
        question.casefold()
        in
        corpus_casefold_set
    )

    overlap_rows.append(
        {
            "human_id":
                idx,

            "human_question":
                question,

            "exact_normalized_overlap":
                exact_match,

            "casefold_normalized_overlap":
                casefold_match,
        }
    )


overlap_df = pd.DataFrame(
    overlap_rows
)


exact_overlap_n = int(
    overlap_df[
        "exact_normalized_overlap"
    ].sum()
)


casefold_overlap_n = int(
    overlap_df[
        "casefold_normalized_overlap"
    ].sum()
)


print("\n" + "=" * 100)
print("OVERLAP / LEAKAGE AUDIT")
print("=" * 100)

print(
    "Exact normalized overlap    =",
    f"{exact_overlap_n}/200"
)

print(
    "Casefold normalized overlap =",
    f"{casefold_overlap_n}/200"
)


if (
    exact_overlap_n == 0
    and
    casefold_overlap_n == 0
):

    print(
        "✅ No normalized overlap."
    )

else:

    print(
        "⚠️ Normalized overlap detected."
    )


# ======================================================================
# 9. CSE RESOURCE LOADERS
# ======================================================================

def clean_resource_text(value):

    if value is None:
        return ""

    return (
        str(value)
        .replace(
            "\ufeff",
            ""
        )
        .strip()
    )


def load_stems(path):

    wb = openpyxl.load_workbook(
        path,
        read_only=True,
        data_only=True
    )

    ws = wb.active

    values = []

    for row in ws.iter_rows(
        min_col=1,
        max_col=1,
        values_only=True
    ):

        value = clean_resource_text(
            row[0]
        )

        if value:
            values.append(
                value
            )

    wb.close()

    return values


def load_jurnaqs(path):

    wb = xlrd.open_workbook(
        path
    )

    sh = wb.sheet_by_index(
        0
    )

    values = []

    # Original resource has a header row.
    for rownum in range(
        1,
        sh.nrows
    ):

        value = clean_resource_text(
            sh.cell(
                rownum,
                0
            ).value
        )

        if value:
            values.append(
                value
            )

    return values


def load_endings(path):

    wb = xlrd.open_workbook(
        path
    )

    sh = wb.sheet_by_index(
        0
    )

    endings = []
    endings_segmented = []

    # Original resource has a header row.
    for rownum in range(
        1,
        sh.nrows
    ):

        ending = clean_resource_text(
            sh.cell(
                rownum,
                0
            ).value
        )

        segmented = clean_resource_text(
            sh.cell(
                rownum,
                1
            ).value
        )

        if ending:

            endings.append(
                ending
            )

            endings_segmented.append(
                segmented
            )

    return (
        endings,
        endings_segmented
    )


def read_stopwords(path):

    candidate_encodings = [
        "utf-8-sig",
        "utf-8",
        "utf-16",
        "utf-16-le",
        "utf-16-be",
        "cp1251",
        "cp1252",
        "latin-1",
    ]

    best = None


    for encoding in candidate_encodings:

        try:

            text = Path(path).read_text(
                encoding=encoding,
                errors="strict"
            )

            null_ratio = (
                text.count(
                    "\x00"
                )
                /
                max(
                    len(text),
                    1
                )
            )

            if null_ratio > 0.01:
                continue


            lines = [
                line.replace(
                    "\ufeff",
                    ""
                ).strip()

                for line
                in text.splitlines()
            ]


            lines = [
                line
                for line
                in lines
                if line
            ]


            if not lines:
                continue


            score = (
                int(
                    len(lines)
                    ==
                    EXPECTED[
                        "stopword_entries"
                    ]
                )
                *
                100000
            )


            score += sum(
                any(
                    ch in line
                    for ch
                    in
                    "әғқңөұүһіӘҒҚҢӨҰҮҺІ"
                )
                for line
                in lines
            )


            candidate = (
                score,
                encoding,
                lines
            )


            if (
                best is None
                or
                candidate[0]
                >
                best[0]
            ):
                best = candidate


        except Exception:
            continue


    if best is None:

        raise RuntimeError(
            "Cannot decode stop_words file."
        )


    _, encoding, lines = best

    return (
        lines,
        encoding
    )


print("\n" + "=" * 100)
print("LOADING RELATIONAL CSE RESOURCES")
print("=" * 100)


stems = load_stems(
    paths["stems"]
)

jurnaqs = load_jurnaqs(
    paths["jurnaqs"]
)

endings, endings_seg = load_endings(
    paths["endings"]
)

stop_words, stopword_encoding = read_stopwords(
    paths["stopwords"]
)


resource_audit = {
    "stems_entries":
        len(
            stems
        ),

    "stems_unique":
        len(
            set(
                stems
            )
        ),

    "jurnaq_entries":
        len(
            jurnaqs
        ),

    "jurnaq_unique":
        len(
            set(
                jurnaqs
            )
        ),

    "ending_mappings":
        len(
            endings
        ),

    "ending_surfaces_unique":
        len(
            set(
                endings
            )
        ),

    "stopword_entries":
        len(
            stop_words
        ),

    "stopword_unique":
        len(
            set(
                stop_words
            )
        ),
}


for key, value in resource_audit.items():

    print(
        f"{key:<28} = "
        f"{value:,}"
    )


print(
    f"{'stopword_encoding':<28} = "
    f"{stopword_encoding}"
)


# Hard validation against Problem-1 resource audit
for key, value in resource_audit.items():

    if (
        value
        !=
        EXPECTED[key]
    ):

        raise RuntimeError(
            f"STOP: resource mismatch for {key}. "
            f"Actual={value}, "
            f"expected={EXPECTED[key]}."
        )


print(
    "✅ CSE resources reproduce the validated "
    "Problem-1 resource counts."
)


# ======================================================================
# 10. FAST CSE LOOKUP STRUCTURES
# ======================================================================

stems_set = set(
    stem.lower()
    for stem
    in stems
)


jurnaqs_set = set(
    jurnaqs
)


endings_set = set(
    endings
)


stop_words_set = set(
    word.lower()
    for word
    in stop_words
)


# Original implementation uses first occurrence of ending mapping.
ending_first_seg = {}


for ending, segmented in zip(
    endings,
    endings_seg
):

    if (
        ending
        not in
        ending_first_seg
    ):

        ending_first_seg[
            ending
        ] = segmented


# ======================================================================
# 11. RELATIONAL CSE FUNCTIONS
#
# Inflectional endings:
#   right-to-left longest-first matching
#
# Derivational suffixes:
#   shortest-first iterative stripping
# ======================================================================

def fjurnaq_stem(word):
    """
    Derivational suffix:
    shortest-first iterative stripping.
    """

    word_len = len(
        word
    )

    min_len_of_stem = 2


    if (
        word_len
        <=
        min_len_of_stem
    ):

        return (
            word,
            ""
        )


    i = 1
    max_len_jurnaq = 4


    while (
        i <= len(word)
        and
        i <= max_len_jurnaq
    ):

        word_ending = word[
            -i:
        ]

        stem = word[
            :-i
        ]


        if (
            word_ending
            in
            jurnaqs_set

            and

            stem.lower()
            in
            stems_set
        ):

            return (
                stem,
                word_ending
            )


        i += 1


    return (
        "",
        ""
    )


def ending_stem(word):
    """
    Inflectional ending:
    right-to-left longest-first matching.
    """

    word_len = len(
        word
    )

    min_len_of_word = 2


    if (
        word_len
        <=
        min_len_of_word
    ):

        return (
            word,
            ""
        )


    if (
        word.lower()
        in
        stems_set
    ):

        return (
            word,
            ""
        )


    n = (
        word_len
        -
        min_len_of_word
    )


    i = (
        n
        +
        1
    )


    rez_stem = ""


    while i > 0:

        word_ending = word[
            word_len
            -
            (i - 1):
        ]


        if word_ending:

            stem = word[
                :
                word_len
                -
                len(
                    word_ending
                )
            ]

        else:

            stem = word


        if (
            word_ending
            in
            endings_set

            and

            stem.lower()
            in
            stems_set
        ):

            return (
                stem,
                word_ending
            )


        rez_stem = stem

        i -= 1


    return (
        rez_stem,
        ""
    )


def segment_word(word):

    stemm, ending = ending_stem(
        word
    )


    if ending:

        endingseg = ending_first_seg.get(
            ending,
            ""
        )

    else:

        endingseg = ""


    jurnaqs_seg = ""


    if (
        len(
            stemm
        )
        >
        2
    ):

        n = (
            len(
                stemm
            )
            -
            2
        )


        i = 1


        while i <= n:

            rez_stem, jurnaq = fjurnaq_stem(
                stemm
            )


            if not jurnaq:
                break


            if jurnaqs_seg:

                jurnaqs_seg = (
                    f"{jurnaq}@@ "
                    f"{jurnaqs_seg}"
                )

            else:

                jurnaqs_seg = jurnaq


            stemm = rez_stem

            i += len(
                jurnaq
            )


    if (
        endingseg
        and
        jurnaqs_seg
    ):

        return (
            f"{stemm}@@ "
            f"{jurnaqs_seg}"
            f"{endingseg}"
        )


    if endingseg:

        return (
            f"{stemm}@@ "
            f"{endingseg}"
        )


    if jurnaqs_seg:

        return (
            f"{stemm}@@ "
            f"{jurnaqs_seg}"
        )


    return stemm


def segment_token(token):

    if (
        token.lower()
        in
        stop_words_set
    ):

        return token


    if token.isalpha():

        return segment_word(
            token
        )


    return token


# ======================================================================
# 12. SPAN-PRESERVING TOKENIZATION
#
# Important:
# We NEVER rebuild a question using " ".join(tokens).
#
# Original punctuation and whitespace spans are preserved exactly.
# ======================================================================

TOKENIZER_RE = re.compile(
    r"\w+|[^\w\s]",
    flags=re.UNICODE
)


all_questions_for_cache = (
    human_norm_questions
    +
    baseline_unique_questions
)


unique_alpha_tokens = []

seen_alpha_tokens = set()


for question in all_questions_for_cache:

    for match in TOKENIZER_RE.finditer(
        question
    ):

        token = match.group(
            0
        )

        if (
            token.isalpha()
            and
            token
            not in
            seen_alpha_tokens
        ):

            seen_alpha_tokens.add(
                token
            )

            unique_alpha_tokens.append(
                token
            )


print("\n" + "=" * 100)
print("BUILDING CSE WORD CACHE")
print("=" * 100)

print(
    "Unique alphabetic surface tokens =",
    f"{len(unique_alpha_tokens):,}"
)


word_seg_cache = {}


for idx, token in enumerate(
    unique_alpha_tokens,
    start=1
):

    word_seg_cache[
        token
    ] = segment_token(
        token
    )


    if (
        idx % 2000 == 0
        or
        idx
        ==
        len(
            unique_alpha_tokens
        )
    ):

        print(
            f"CSE word cache: "
            f"{idx:,}/"
            f"{len(unique_alpha_tokens):,}"
        )


# ======================================================================
# 13. MORPHOLOGY METRICS
# ======================================================================

MORPH_METRICS = [
    (
        "boundaries_per_question",
        "CSE boundaries / question"
    ),

    (
        "boundaries_per_word",
        "CSE boundaries / alphabetic word"
    ),

    (
        "complex_word_proportion",
        "Complex-word proportion"
    ),

    (
        "mean_word_length",
        "Mean alphabetic word length"
    ),

    (
        "max_boundaries_per_word",
        "Max CSE boundaries / word"
    ),

    (
        "alphabetic_words_per_question",
        "Alphabetic words / question"
    ),
]


def reconstruct_segmented_token(
    segmented_token
):
    """
    Remove CSE markers only INSIDE a single token.

    This prevents a trailing '@@ ' sequence from consuming the
    original inter-word space.
    """

    return (
        str(
            segmented_token
        )
        .replace(
            "@@ ",
            ""
        )
        .replace(
            "@@",
            ""
        )
    )


def question_morphology_metrics(
    question
):

    # Use the same clean-normalized question representation
    # for both Human200 and MainCorpus14689.
    question = norm_space_punct(
        question
    )


    alpha_words = []
    boundary_counts = []


    segmented_pieces = []
    reconstructed_pieces = []


    token_mapping_mismatch_count = 0
    token_mapping_mismatch_examples = []


    last_end = 0


    for match in TOKENIZER_RE.finditer(
        question
    ):

        # ----------------------------------------------
        # Preserve text between tokens exactly.
        # ----------------------------------------------

        original_separator = question[
            last_end:
            match.start()
        ]


        segmented_pieces.append(
            original_separator
        )

        reconstructed_pieces.append(
            original_separator
        )


        token = match.group(
            0
        )


        # ----------------------------------------------
        # Alphabetic token: apply CSE
        # ----------------------------------------------

        if token.isalpha():

            alpha_words.append(
                token
            )


            segmented_token = word_seg_cache[
                token
            ]


            boundary_count = segmented_token.count(
                "@@"
            )


            boundary_counts.append(
                boundary_count
            )


            reconstructed_token = reconstruct_segmented_token(
                segmented_token
            )


            segmented_pieces.append(
                segmented_token
            )


            reconstructed_pieces.append(
                reconstructed_token
            )


            # Resource-level non-reversible mapping diagnostic
            if (
                reconstructed_token
                !=
                token
            ):

                token_mapping_mismatch_count += 1


                if (
                    len(
                        token_mapping_mismatch_examples
                    )
                    <
                    5
                ):

                    token_mapping_mismatch_examples.append(
                        {
                            "original":
                                token,

                            "segmented":
                                segmented_token,

                            "reconstructed":
                                reconstructed_token,
                        }
                    )


        # ----------------------------------------------
        # Punctuation / operators / code notation:
        # preserve unchanged.
        # ----------------------------------------------

        else:

            segmented_pieces.append(
                token
            )

            reconstructed_pieces.append(
                token
            )


        last_end = match.end()


    # Preserve tail exactly.
    original_tail = question[
        last_end:
    ]


    segmented_pieces.append(
        original_tail
    )

    reconstructed_pieces.append(
        original_tail
    )


    segmented_question = "".join(
        segmented_pieces
    )


    reconstructed_question = "".join(
        reconstructed_pieces
    )


    exact_reconstruction_match = (
        reconstructed_question
        ==
        question
    )


    n_words = len(
        alpha_words
    )


    total_boundaries = int(
        sum(
            boundary_counts
        )
    )


    if n_words > 0:

        boundaries_per_word = (
            total_boundaries
            /
            n_words
        )


        complex_word_proportion = (
            sum(
                1
                for count
                in boundary_counts
                if count > 0
            )
            /
            n_words
        )


        mean_word_length = float(
            np.mean(
                [
                    len(
                        word
                    )
                    for word
                    in alpha_words
                ]
            )
        )


        max_boundaries = float(
            max(
                boundary_counts
            )
        )


    else:

        boundaries_per_word = 0.0
        complex_word_proportion = 0.0
        mean_word_length = 0.0
        max_boundaries = 0.0


    return {
        "question":
            question,

        "segmented_question":
            segmented_question,

        "reconstructed_question":
            reconstructed_question,

        "reconstruction_match":
            exact_reconstruction_match,

        "token_mapping_mismatch_count":
            int(
                token_mapping_mismatch_count
            ),

        "token_mapping_mismatch_examples":
            json.dumps(
                token_mapping_mismatch_examples,
                ensure_ascii=False
            ),

        "boundaries_per_question":
            float(
                total_boundaries
            ),

        "boundaries_per_word":
            float(
                boundaries_per_word
            ),

        "complex_word_proportion":
            float(
                complex_word_proportion
            ),

        "mean_word_length":
            float(
                mean_word_length
            ),

        "max_boundaries_per_word":
            float(
                max_boundaries
            ),

        "alphabetic_words_per_question":
            float(
                n_words
            ),
    }


def analyze_question_set(
    questions,
    group_name
):

    rows = []

    total = len(
        questions
    )


    for idx, question in enumerate(
        questions,
        start=1
    ):

        metrics = question_morphology_metrics(
            question
        )


        metrics[
            "group"
        ] = group_name


        metrics[
            "item_id"
        ] = idx


        rows.append(
            metrics
        )


        if (
            idx % 1000 == 0
            or
            idx == total
        ):

            print(
                f"{group_name}: "
                f"{idx:,}/"
                f"{total:,}"
            )


    return pd.DataFrame(
        rows
    )


# ======================================================================
# 14. RUN MORPHOLOGY AUDIT
# ======================================================================

print("\n" + "=" * 100)
print("MORPHOLOGY AUDIT — HUMAN 200")
print("=" * 100)


human_metrics_df = analyze_question_set(
    human_norm_questions,
    "Human200"
)


print("\n" + "=" * 100)
print("MORPHOLOGY AUDIT — MAIN CLEAN CORPUS")
print("=" * 100)


corpus_metrics_df = analyze_question_set(
    baseline_unique_questions,
    "MainCorpus14689"
)


# ======================================================================
# 15. RECONSTRUCTION DIAGNOSTIC
#
# NOT a hard validity criterion.
# ======================================================================

human_reconstruction_n = int(
    human_metrics_df[
        "reconstruction_match"
    ].sum()
)


corpus_reconstruction_n = int(
    corpus_metrics_df[
        "reconstruction_match"
    ].sum()
)


human_reconstruction_rate = (
    human_reconstruction_n
    /
    len(
        human_metrics_df
    )
)


corpus_reconstruction_rate = (
    corpus_reconstruction_n
    /
    len(
        corpus_metrics_df
    )
)


human_token_mapping_mismatches = int(
    human_metrics_df[
        "token_mapping_mismatch_count"
    ].sum()
)


corpus_token_mapping_mismatches = int(
    corpus_metrics_df[
        "token_mapping_mismatch_count"
    ].sum()
)


human_bad = human_metrics_df[
    ~human_metrics_df[
        "reconstruction_match"
    ]
].copy()


corpus_bad = corpus_metrics_df[
    ~corpus_metrics_df[
        "reconstruction_match"
    ]
].copy()


print("\n" + "=" * 100)
print("CSE RECONSTRUCTION DIAGNOSTIC")
print("=" * 100)


print(
    "Human exact reconstruction =",
    f"{human_reconstruction_n}/"
    f"{len(human_metrics_df)}"
)


print(
    "Corpus exact reconstruction =",
    f"{corpus_reconstruction_n}/"
    f"{len(corpus_metrics_df)}"
)


print(
    "Human exact reconstruction rate =",
    f"{human_reconstruction_rate:.9f}"
)


print(
    "Corpus exact reconstruction rate =",
    f"{corpus_reconstruction_rate:.9f}"
)


print(
    "Human token mapping mismatches =",
    human_token_mapping_mismatches
)


print(
    "Corpus token mapping mismatches =",
    corpus_token_mapping_mismatches
)


if len(
    human_bad
) > 0:

    print(
        "\nHUMAN NON-REVERSIBLE EXAMPLES:"
    )

    print(
        human_bad[
            [
                "item_id",
                "question",
                "segmented_question",
                "reconstructed_question",
                "token_mapping_mismatch_count",
            ]
        ]
        .head(20)
        .to_string(
            index=False
        )
    )


if len(
    corpus_bad
) > 0:

    print(
        "\nCORPUS NON-REVERSIBLE EXAMPLES:"
    )

    print(
        corpus_bad[
            [
                "item_id",
                "question",
                "segmented_question",
                "reconstructed_question",
                "token_mapping_mismatch_count",
            ]
        ]
        .head(20)
        .to_string(
            index=False
        )
    )


print(
    "\nNOTE: exact reconstruction is a diagnostic only."
)


print(
    "The morphology-complexity analysis uses the original clean "
    "questions and counts detected CSE boundaries. "
    "Rare resource-level surface-form substitutions therefore "
    "do not invalidate the morphology statistics."
)


print(
    "✅ Reconstruction diagnostic completed."
)


# ======================================================================
# 16. DESCRIPTIVE STATISTICS
# ======================================================================

def descriptive_row(
    values,
    group,
    metric,
    label
):

    x = np.asarray(
        values,
        dtype=float
    )


    return {
        "group":
            group,

        "metric":
            metric,

        "metric_label":
            label,

        "N":
            len(
                x
            ),

        "mean":
            float(
                np.mean(
                    x
                )
            ),

        "SD":
            float(
                np.std(
                    x,
                    ddof=1
                )
            ),

        "median":
            float(
                np.median(
                    x
                )
            ),

        "Q1":
            float(
                np.quantile(
                    x,
                    0.25
                )
            ),

        "Q3":
            float(
                np.quantile(
                    x,
                    0.75
                )
            ),

        "min":
            float(
                np.min(
                    x
                )
            ),

        "max":
            float(
                np.max(
                    x
                )
            ),
    }


descriptive_rows = []


for metric, label in MORPH_METRICS:

    descriptive_rows.append(
        descriptive_row(
            human_metrics_df[
                metric
            ],
            "Human200",
            metric,
            label
        )
    )


    descriptive_rows.append(
        descriptive_row(
            corpus_metrics_df[
                metric
            ],
            "MainCorpus14689",
            metric,
            label
        )
    )


descriptive_df = pd.DataFrame(
    descriptive_rows
)


print("\n" + "=" * 100)
print("DESCRIPTIVE MORPHOLOGY STATISTICS")
print("=" * 100)

print(
    descriptive_df.to_string(
        index=False
    )
)


# ======================================================================
# 17. EFFECT SIZE — HEDGES' g
# ======================================================================

def hedges_g_independent(
    human,
    corpus
):

    x = np.asarray(
        human,
        dtype=float
    )

    y = np.asarray(
        corpus,
        dtype=float
    )


    n1 = len(
        x
    )

    n2 = len(
        y
    )


    var1 = np.var(
        x,
        ddof=1
    )

    var2 = np.var(
        y,
        ddof=1
    )


    pooled_variance = (
        (
            (n1 - 1)
            *
            var1
        )
        +
        (
            (n2 - 1)
            *
            var2
        )
    ) / (
        n1
        +
        n2
        -
        2
    )


    if pooled_variance <= 0:

        return 0.0


    pooled_sd = math.sqrt(
        pooled_variance
    )


    d = (
        np.mean(
            x
        )
        -
        np.mean(
            y
        )
    ) / pooled_sd


    df = (
        n1
        +
        n2
        -
        2
    )


    correction = (
        1
        -
        3
        /
        (
            4
            *
            df
            -
            1
        )
    )


    return float(
        correction
        *
        d
    )


# ======================================================================
# 18. BOOTSTRAP 95% CI
#
# Difference = Human200 - MainCorpus14689
# ======================================================================

def independent_bootstrap_diff_ci(
    human,
    corpus,
    seed,
    n_boot=N_BOOT,
    chunk_size=100
):

    x = np.asarray(
        human,
        dtype=float
    )

    y = np.asarray(
        corpus,
        dtype=float
    )


    observed = float(
        np.mean(
            x
        )
        -
        np.mean(
            y
        )
    )


    rng = np.random.default_rng(
        seed
    )


    n1 = len(
        x
    )

    n2 = len(
        y
    )


    differences = np.empty(
        n_boot,
        dtype=float
    )


    pos = 0


    while pos < n_boot:

        current = min(
            chunk_size,
            n_boot - pos
        )


        index_x = rng.integers(
            0,
            n1,
            size=(
                current,
                n1
            )
        )


        index_y = rng.integers(
            0,
            n2,
            size=(
                current,
                n2
            )
        )


        mean_x = np.mean(
            x[
                index_x
            ],
            axis=1
        )


        mean_y = np.mean(
            y[
                index_y
            ],
            axis=1
        )


        differences[
            pos:
            pos + current
        ] = (
            mean_x
            -
            mean_y
        )


        pos += current


    ci_low, ci_high = np.quantile(
        differences,
        [
            0.025,
            0.975,
        ]
    )


    return (
        observed,
        float(
            ci_low
        ),
        float(
            ci_high
        )
    )


# ======================================================================
# 19. TWO-SIDED INDEPENDENT PERMUTATION TEST
# ======================================================================

def independent_permutation_p(
    human,
    corpus,
    seed,
    n_perm=N_PERM
):

    x = np.asarray(
        human,
        dtype=float
    )

    y = np.asarray(
        corpus,
        dtype=float
    )


    pooled = np.concatenate(
        [
            x,
            y
        ]
    )


    n1 = len(
        x
    )


    n_total = len(
        pooled
    )


    observed = abs(
        float(
            np.mean(
                x
            )
            -
            np.mean(
                y
            )
        )
    )


    total_sum = float(
        np.sum(
            pooled
        )
    )


    rng = np.random.default_rng(
        seed
    )


    all_indices = np.arange(
        n_total
    )


    extreme = 0


    for _ in range(
        n_perm
    ):

        selected = rng.choice(
            all_indices,
            size=n1,
            replace=False
        )


        selected_sum = float(
            np.sum(
                pooled[
                    selected
                ]
            )
        )


        mean_group1 = (
            selected_sum
            /
            n1
        )


        mean_group2 = (
            total_sum
            -
            selected_sum
        ) / (
            n_total
            -
            n1
        )


        permutation_difference = abs(
            mean_group1
            -
            mean_group2
        )


        if (
            permutation_difference
            >=
            observed
            -
            1e-15
        ):

            extreme += 1


    return float(
        (
            extreme
            +
            1
        )
        /
        (
            n_perm
            +
            1
        )
    )


# ======================================================================
# 20. HOLM MULTIPLE-TESTING CORRECTION
# ======================================================================

def holm_adjust(
    p_values
):

    p = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p
    )


    order = np.argsort(
        p
    )


    adjusted_sorted = np.empty(
        m,
        dtype=float
    )


    running_max = 0.0


    for rank, idx in enumerate(
        order
    ):

        candidate = (
            (
                m
                -
                rank
            )
            *
            p[
                idx
            ]
        )


        running_max = max(
            running_max,
            candidate
        )


        adjusted_sorted[
            rank
        ] = min(
            1.0,
            running_max
        )


    adjusted = np.empty(
        m,
        dtype=float
    )


    for rank, idx in enumerate(
        order
    ):

        adjusted[
            idx
        ] = adjusted_sorted[
            rank
        ]


    return adjusted


# ======================================================================
# 21. PRIMARY MORPHOLOGY STATISTICS
# ======================================================================

print("\n" + "=" * 100)
print("STATISTICAL COMPARISON")
print("Difference = Human200 - MainCorpus14689")
print("=" * 100)


stat_rows = []


for metric_index, (
    metric,
    metric_label
) in enumerate(
    MORPH_METRICS
):

    print(
        f"[{metric_index + 1}/"
        f"{len(MORPH_METRICS)}] "
        f"{metric_label}"
    )


    human_values = human_metrics_df[
        metric
    ].to_numpy(
        dtype=float
    )


    corpus_values = corpus_metrics_df[
        metric
    ].to_numpy(
        dtype=float
    )


    (
        delta,
        ci_low,
        ci_high
    ) = independent_bootstrap_diff_ci(
        human_values,
        corpus_values,
        seed=(
            STAT_SEED
            +
            metric_index
        ),
        n_boot=N_BOOT
    )


    p_perm = independent_permutation_p(
        human_values,
        corpus_values,
        seed=(
            STAT_SEED
            +
            1000
            +
            metric_index
        ),
        n_perm=N_PERM
    )


    hedges_g = hedges_g_independent(
        human_values,
        corpus_values
    )


    # Secondary descriptive robustness statistic.
    # Not used for the primary Holm family.
    welch = stats.ttest_ind(
        human_values,
        corpus_values,
        equal_var=False,
        alternative="two-sided"
    )


    stat_rows.append(
        {
            "metric":
                metric,

            "metric_label":
                metric_label,

            "Human_N":
                len(
                    human_values
                ),

            "Corpus_N":
                len(
                    corpus_values
                ),

            "Human_mean":
                float(
                    np.mean(
                        human_values
                    )
                ),

            "Corpus_mean":
                float(
                    np.mean(
                        corpus_values
                    )
                ),

            "Delta_Human_minus_Corpus":
                float(
                    delta
                ),

            "CI95_low":
                float(
                    ci_low
                ),

            "CI95_high":
                float(
                    ci_high
                ),

            "Permutation_p_raw":
                float(
                    p_perm
                ),

            "Welch_t":
                float(
                    welch.statistic
                ),

            "Welch_p":
                float(
                    welch.pvalue
                ),

            "Hedges_g":
                float(
                    hedges_g
                ),
        }
    )


stats_df = pd.DataFrame(
    stat_rows
)


stats_df[
    "Permutation_p_Holm6"
] = holm_adjust(
    stats_df[
        "Permutation_p_raw"
    ].to_numpy()
)


stats_df[
    "Significant_Holm6"
] = (
    stats_df[
        "Permutation_p_Holm6"
    ]
    <
    ALPHA
)


stats_df[
    "Direction"
] = np.where(
    stats_df[
        "Delta_Human_minus_Corpus"
    ]
    >
    0,

    "Human200 higher",

    np.where(
        stats_df[
            "Delta_Human_minus_Corpus"
        ]
        <
        0,

        "Human200 lower",

        "No difference"
    )
)


stats_df[
    "CI95"
] = stats_df.apply(
    lambda row:
        (
            f"["
            f"{row['CI95_low']:.6f}, "
            f"{row['CI95_high']:.6f}"
            f"]"
        ),
    axis=1
)


display_stats_columns = [
    "metric_label",

    "Human_N",

    "Corpus_N",

    "Human_mean",

    "Corpus_mean",

    "Delta_Human_minus_Corpus",

    "CI95",

    "Permutation_p_raw",

    "Permutation_p_Holm6",

    "Significant_Holm6",

    "Hedges_g",

    "Direction",
]


print("\n" + "=" * 100)
print("TABLE A. HUMAN 200 VS MAIN CORPUS MORPHOLOGY")
print("=" * 100)


print(
    stats_df[
        display_stats_columns
    ].to_string(
        index=False
    )
)


# ======================================================================
# 22. ADDITIONAL BOUNDARY-DISTRIBUTION AUDIT
# ======================================================================

def boundary_distribution(
    df,
    group_name
):

    n = len(
        df
    )


    any_boundary = int(
        (
            df[
                "boundaries_per_question"
            ]
            >
            0
        ).sum()
    )


    three_plus = int(
        (
            df[
                "boundaries_per_question"
            ]
            >=
            3
        ).sum()
    )


    five_plus = int(
        (
            df[
                "boundaries_per_question"
            ]
            >=
            5
        ).sum()
    )


    return {
        "group":
            group_name,

        "N":
            n,

        "questions_with_any_boundary":
            any_boundary,

        "any_boundary_rate":
            any_boundary
            /
            n,

        "questions_with_3plus_boundaries":
            three_plus,

        "three_plus_rate":
            three_plus
            /
            n,

        "questions_with_5plus_boundaries":
            five_plus,

        "five_plus_rate":
            five_plus
            /
            n,
    }


boundary_df = pd.DataFrame(
    [
        boundary_distribution(
            human_metrics_df,
            "Human200"
        ),

        boundary_distribution(
            corpus_metrics_df,
            "MainCorpus14689"
        ),
    ]
)


print("\n" + "=" * 100)
print("BOUNDARY-DISTRIBUTION AUDIT")
print("=" * 100)


print(
    boundary_df.to_string(
        index=False
    )
)


# ======================================================================
# 23. HUMAN CSE EXAMPLES
# ======================================================================

example_columns = [
    "item_id",

    "question",

    "segmented_question",

    "reconstructed_question",

    "reconstruction_match",

    "token_mapping_mismatch_count",

    "boundaries_per_question",

    "boundaries_per_word",

    "complex_word_proportion",

    "mean_word_length",

    "max_boundaries_per_word",

    "alphabetic_words_per_question",
]


examples_df = human_metrics_df[
    example_columns
].head(
    20
).copy()


print("\n" + "=" * 100)
print("FIRST 10 HUMAN-EVALUATION CSE EXAMPLES")
print("=" * 100)


print(
    examples_df
    .head(
        10
    )
    .to_string(
        index=False
    )
)


# ======================================================================
# 24. INTERPRETATION SUPPORT
# ======================================================================

def effect_magnitude(
    g
):

    value = abs(
        float(
            g
        )
    )


    if value < 0.20:
        return "very small"

    elif value < 0.50:
        return "small"

    elif value < 0.80:
        return "moderate"

    else:
        return "large"


interpretation_rows = []


for _, row in stats_df.iterrows():

    interpretation_rows.append(
        {
            "metric":
                row[
                    "metric_label"
                ],

            "direction":
                row[
                    "Direction"
                ],

            "significant_after_Holm":
                bool(
                    row[
                        "Significant_Holm6"
                    ]
                ),

            "Hedges_g":
                float(
                    row[
                        "Hedges_g"
                    ]
                ),

            "effect_magnitude":
                effect_magnitude(
                    row[
                        "Hedges_g"
                    ]
                ),
        }
    )


interpretation_df = pd.DataFrame(
    interpretation_rows
)


# ======================================================================
# 25. SAVE OUTPUT FILES
# ======================================================================

overlap_path = (
    OUT_DIR
    /
    "problem5_overlap_audit_200_vs_14689.csv"
)


human_metrics_path = (
    OUT_DIR
    /
    "problem5_human200_item_morphology_metrics.csv"
)


corpus_metrics_path = (
    OUT_DIR
    /
    "problem5_corpus14689_item_morphology_metrics.csv"
)


reconstruction_path = (
    OUT_DIR
    /
    "problem5_reconstruction_diagnostic.csv"
)


descriptive_path = (
    OUT_DIR
    /
    "problem5_morphology_descriptive_statistics.csv"
)


statistics_path = (
    OUT_DIR
    /
    "problem5_morphology_inferential_statistics.csv"
)


boundary_path = (
    OUT_DIR
    /
    "problem5_boundary_distribution_audit.csv"
)


examples_path = (
    OUT_DIR
    /
    "problem5_human_CSE_examples.csv"
)


resource_path = (
    OUT_DIR
    /
    "problem5_CSE_resource_audit.csv"
)


environment_path = (
    OUT_DIR
    /
    "problem5_environment.json"
)


overlap_df.to_csv(
    overlap_path,
    index=False,
    encoding="utf-8-sig"
)


human_metrics_df.to_csv(
    human_metrics_path,
    index=False,
    encoding="utf-8-sig"
)


corpus_metrics_df.to_csv(
    corpus_metrics_path,
    index=False,
    encoding="utf-8-sig"
)


reconstruction_df = pd.concat(
    [
        human_bad.assign(
            dataset="Human200"
        ),

        corpus_bad.assign(
            dataset="MainCorpus14689"
        ),
    ],
    ignore_index=True
)


reconstruction_df.to_csv(
    reconstruction_path,
    index=False,
    encoding="utf-8-sig"
)


descriptive_df.to_csv(
    descriptive_path,
    index=False,
    encoding="utf-8-sig"
)


stats_df.to_csv(
    statistics_path,
    index=False,
    encoding="utf-8-sig"
)


boundary_df.to_csv(
    boundary_path,
    index=False,
    encoding="utf-8-sig"
)


examples_df.to_csv(
    examples_path,
    index=False,
    encoding="utf-8-sig"
)


resource_audit_df = pd.DataFrame(
    [
        {
            **resource_audit,

            "stopword_encoding":
                stopword_encoding,
        }
    ]
)


resource_audit_df.to_csv(
    resource_path,
    index=False,
    encoding="utf-8-sig"
)


# ======================================================================
# 26. ENVIRONMENT / REPRODUCIBILITY RECORD
# ======================================================================

environment = {
    "reviewer":
        "Reviewer 1",

    "problem":
        5,

    "analysis":
        (
            "Independent external human-evaluation "
            "morphology complexity audit"
        ),

    "human_evaluation_set":
        (
            "independently prepared external "
            "clean questions"
        ),

    "human_questions":
        int(
            len(
                human_metrics_df
            )
        ),

    "human_unique_questions":
        int(
            human_unique_exact
        ),

    "prospective_numeric_morphology_threshold":
        False,

    "morphology_analysis":
        "post-hoc audit",

    "main_baseline_raw_records":
        int(
            len(
                baseline_rows
            )
        ),

    "main_unique_clean_questions":
        int(
            len(
                baseline_unique_questions
            )
        ),

    "overlap": {
        "exact_normalized":
            int(
                exact_overlap_n
            ),

        "casefold_normalized":
            int(
                casefold_overlap_n
            ),

        "fuzzy_matching":
            False,
    },

    "CSE_reconstruction_diagnostic": {
        "Human200_exact":
            (
                f"{human_reconstruction_n}/"
                f"{len(human_metrics_df)}"
            ),

        "MainCorpus14689_exact":
            (
                f"{corpus_reconstruction_n}/"
                f"{len(corpus_metrics_df)}"
            ),

        "Human200_rate":
            float(
                human_reconstruction_rate
            ),

        "MainCorpus14689_rate":
            float(
                corpus_reconstruction_rate
            ),

        "Human_token_mapping_mismatches":
            int(
                human_token_mapping_mismatches
            ),

        "Corpus_token_mapping_mismatches":
            int(
                corpus_token_mapping_mismatches
            ),

        "interpretation":
            (
                "Diagnostic only. Exact surface regeneration "
                "is not required for morphology-complexity "
                "analysis because the original clean question "
                "is retained and CSE is an internal "
                "preprocessing representation."
            ),
    },

    "CSE_matching_rules": {
        "inflectional_endings":
            (
                "right-to-left longest-first matching"
            ),

        "derivational_suffixes":
            (
                "shortest-first iterative stripping"
            ),
    },

    "CSE_resources":
        resource_audit,

    "stopword_encoding":
        stopword_encoding,

    "morphology_metrics":
        [
            label
            for _, label
            in MORPH_METRICS
        ],

    "statistics": {
        "bootstrap_replicates":
            int(
                N_BOOT
            ),

        "permutation_replicates":
            int(
                N_PERM
            ),

        "statistical_seed":
            int(
                STAT_SEED
            ),

        "alpha":
            float(
                ALPHA
            ),

        "primary_test":
            (
                "two-sided independent "
                "permutation test"
            ),

        "multiple_testing":
            (
                "Holm correction across "
                "six morphology metrics"
            ),

        "effect_size":
            "Hedges' g",

        "difference_direction":
            (
                "Human200 minus "
                "MainCorpus14689"
            ),
    },

    "input_file_SHA256":
        file_hashes,
}


environment_path.write_text(
    json.dumps(
        environment,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ======================================================================
# 27. ZIP ALL OUTPUTS
# ======================================================================

if ZIP_PATH.exists():
    ZIP_PATH.unlink()


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:

    for output_path in OUT_DIR.iterdir():

        if output_path.is_file():

            archive.write(
                output_path,
                arcname=output_path.name
            )


# ======================================================================
# 28. FINAL COPY BLOCK
# ======================================================================

print("\n\n" + "=" * 100)
print("COPY THIS BLOCK BACK TO CHATGPT")
print("=" * 100)


print(
    "REVIEWER 1 / PROBLEM 5"
)


print(
    "Human-evaluation set = independently prepared "
    "external clean questions"
)


print(
    "Prospective numerical morphology threshold = NO"
)


print(
    "Morphology complexity comparison = POST-HOC audit"
)


print(
    f"\nHuman questions = "
    f"{len(human_metrics_df)}"
)


print(
    f"Human unique questions = "
    f"{human_unique_exact}"
)


print(
    f"Main corpus raw records = "
    f"{len(baseline_rows)}"
)


print(
    f"Main corpus unique Clean questions = "
    f"{len(baseline_unique_questions)}"
)


print(
    "\nOVERLAP / LEAKAGE:"
)


print(
    f"Exact normalized overlap = "
    f"{exact_overlap_n}/200"
)


print(
    f"Casefold normalized overlap = "
    f"{casefold_overlap_n}/200"
)


print(
    "Fuzzy matching used = NO"
)


print(
    "\nCSE RESOURCE AUDIT:"
)


for key, value in resource_audit.items():

    print(
        f"{key} = {value}"
    )


print(
    f"stopword_encoding = "
    f"{stopword_encoding}"
)


print(
    "\nCSE RECONSTRUCTION DIAGNOSTIC:"
)


print(
    f"Human exact reconstruction = "
    f"{human_reconstruction_n}/"
    f"{len(human_metrics_df)}"
)


print(
    f"Human reconstruction rate = "
    f"{human_reconstruction_rate:.9f}"
)


print(
    f"Main corpus exact reconstruction = "
    f"{corpus_reconstruction_n}/"
    f"{len(corpus_metrics_df)}"
)


print(
    f"Main corpus reconstruction rate = "
    f"{corpus_reconstruction_rate:.9f}"
)


print(
    f"Human token mapping mismatches = "
    f"{human_token_mapping_mismatches}"
)


print(
    f"Corpus token mapping mismatches = "
    f"{corpus_token_mapping_mismatches}"
)


print(
    "Reconstruction interpretation = "
    "DIAGNOSTIC ONLY; not a validity criterion "
    "for morphology-complexity statistics"
)


print(
    "\nMORPHOLOGY COMPARISON:"
)


print(
    stats_df[
        display_stats_columns
    ].to_string(
        index=False
    )
)


print(
    "\nDESCRIPTIVE STATISTICS:"
)


print(
    descriptive_df.to_string(
        index=False
    )
)


print(
    "\nBOUNDARY DISTRIBUTION:"
)


print(
    boundary_df.to_string(
        index=False
    )
)


print(
    "\nINTERPRETATION SUPPORT:"
)


print(
    interpretation_df.to_string(
        index=False
    )
)


print(
    "\nSTATISTICAL PROTOCOL:"
)


print(
    f"Bootstrap replicates = "
    f"{N_BOOT}"
)


print(
    f"Permutation replicates = "
    f"{N_PERM}"
)


print(
    f"Statistical seed = "
    f"{STAT_SEED}"
)


print(
    "Primary p-value = two-sided "
    "independent permutation test"
)


print(
    "Correction = Holm across "
    "6 morphology metrics"
)


print(
    "Effect size = Hedges' g"
)


print(
    "Delta direction = "
    "Human200 - MainCorpus14689"
)


print(
    "\nFILES SAVED:"
)


for output_path in [
    overlap_path,
    human_metrics_path,
    corpus_metrics_path,
    reconstruction_path,
    descriptive_path,
    statistics_path,
    boundary_path,
    examples_path,
    resource_path,
    environment_path,
    ZIP_PATH,
]:

    print(
        output_path
    )


# ======================================================================
# 29. FINAL VALIDITY CHECK
#
# Reconstruction is deliberately NOT included here because it is
# diagnostic rather than a validity criterion for Problem 5.
# ======================================================================

core_checks_passed = (
    len(
        human_metrics_df
    )
    ==
    200

    and

    human_unique_exact
    ==
    200

    and

    len(
        baseline_rows
    )
    ==
    14991

    and

    len(
        baseline_unique_questions
    )
    ==
    14689

    and

    exact_overlap_n
    ==
    0

    and

    casefold_overlap_n
    ==
    0
)


if core_checks_passed:

    print(
        "\n✅ ALL CORE DATA / OVERLAP / "
        "RESOURCE AUDITS PASSED."
    )

else:

    raise RuntimeError(
        "STOP: one or more core validity "
        "checks failed."
    )


print(
    "\n✅ REVIEWER 1 / PROBLEM 5 "
    "ANALYSIS COMPLETED SUCCESSFULLY"
)

INSTALLING / CHECKING PACKAGES

SELECTED INPUT FILES
human        -> /content/200 сұрақ.json
baseline     -> /content/baseline_15000.json
stems        -> /content/qaz_stems_unik_edit.xlsx
jurnaqs      -> /content/qaz_jurnaks.xls
endings      -> /content/qaz_endings_seg.xls
stopwords    -> /content/stop_words.txt

QUESTION DATA AUDIT
Baseline raw records       = 14,991
Human-evaluation questions = 200
Unique Clean corpus Qs     = 14,689
Human unique, exact        = 200
Human unique, casefold     = 200

OVERLAP / LEAKAGE AUDIT
Exact normalized overlap    = 0/200
Casefold normalized overlap = 0/200
✅ No normalized overlap.

LOADING RELATIONAL CSE RESOURCES
stems_entries                = 103,624
stems_unique                 = 103,623
jurnaq_entries               = 146
jurnaq_unique                = 101
ending_mappings              = 3,316
ending_surfaces_unique       = 3,030
stopword_entries             = 189
stopword_unique              = 188
stopword_encoding            = utf-8-sig
✅ CSE